# Mini Project 1 — Analysis Notebook

**Your name: Lina (Jiayu) Wang**  
**Dataset: https://zenodo.org/records/18706837**  
**Date: 05/06/2026**  

This notebook has four sections. Work through them in order. Each section has instructions and a code cell to fill in. Add markdown cells to explain your thinking as you go — that writing is part of the assignment.

When you're done, publish this notebook to your GitHub repository and submit the URL to Canvas.

In [ ]:
!pip install jupyter plotly kaleido pandas -q

import os
import pandas as pd
import plotly.express as px

print("Setup complete.")

---

## Section 1 — Overview

**Dataset:** A Dataset of University Students' Stress and Anxiety Levels based on Questionnaires and Wearable Sensors (SSAQS). Garcia-Ceja et al. (2026). Downloaded from Kaggle as a ZIP archive and extracted locally. The dataset contains real, sensor-collected data from 35 university students across 3 courses (A1, A2, B). Each student has a dedicated folder (numbered 1–35) containing up to 7 separate CSV files representing different data streams collected over the semester: `activity_level.csv`, `daily_questions.csv`, `hrv.csv`, `oxygen.csv`, `sleep.csv`, `steps.csv`, and `stress.csv`.

**Why this dataset:** College students face high and often invisible levels of stress, and this dataset provides a rare combination of real wearable sensor data and self-reported surveys from actual students — making any patterns directly applicable to designing tools or interventions for student wellbeing.

**Three analytical questions:**

1. **Set the stage — How does stress differ by course context?** How do average self-reported stress and anxiety scores (from `daily_questions.csv`) compare across the three course groups (A1, A2, B)? This establishes whether course environment is a meaningful factor before testing behavioral predictors.

2. **Physical behavior as a predictor — Does being more active associate with lower stress?** On days when a student has a higher proportion of active minutes (LIGHTLY_ACTIVE, FAIRLY_ACTIVE, VERY_ACTIVE from `activity_level.csv`), do they also report lower stress scores in `daily_questions.csv`?

3. **Recovery behavior as a predictor — Does sleeping better associate with lower stress?** On days following a higher sleep `overall_score` (`sleep.csv`), do students report lower stress and anxiety in `daily_questions.csv`? Comparing Q2 and Q3 reveals which daily behavior — activity or sleep — shows a stronger association with student stress.

**What a practitioner would do with these findings:** A student wellbeing program or app designer could use these findings to prioritize which behavioral intervention — sleep hygiene vs. physical activity nudges — is more likely to reduce self-reported stress in college students.

---

## Section 2 — Data Profile

Load your dataset and get a basic picture of what's in it. Answer these questions in a markdown cell below your code:

- How many rows and columns does your dataset have?
- What does each column represent?
- Are there any obvious data quality issues (missing values, unexpected types, inconsistent formatting)?
- Which column or columns will your analysis focus on, and why?

In [47]:
import os
import pandas as pd

BASE = "SSAQS dataset"

# Load course mapping (encoding='utf-8-sig' strips the byte-order mark)
users_courses = pd.read_csv(f"{BASE}/users-courses.csv", encoding='utf-8-sig')[['userid', 'course']]

# Students 3, 12, 14 only have daily_questions — excluded from sensor analyses
sensor_incomplete = {3, 12, 14}
print("Sensor-incomplete students (daily_questions only):", sensor_incomplete)

# --- Load daily_questions for all 35 students ---
dq_frames = []
for sid in range(1, 36):
    path = f"{BASE}/{sid}/daily_questions.csv"
    if os.path.exists(path):
        df = pd.read_csv(path)
        df['student_id'] = sid
        dq_frames.append(df)

dq_all = pd.concat(dq_frames, ignore_index=True)

# Timestamps are Unix epoch seconds — convert to UTC date
dq_all['date'] = pd.to_datetime(dq_all['timeStampStart'], unit='s', utc=True).dt.date
dq_all = dq_all[['student_id', 'date', 'stress', 'anxiety']]

# Attach course label
dq_all = dq_all.merge(users_courses, left_on='student_id', right_on='userid').drop(columns='userid')

print(f"\ndaily_questions: {len(dq_all)} survey rows from {dq_all['student_id'].nunique()} students")
dq_all.head()

Sensor-incomplete students (daily_questions only): {3, 12, 14}

daily_questions: 3133 survey rows from 35 students


,student_id,date,stress,anxiety,course
0,1,2025-02-19,15,0,A1
1,1,2025-02-20,33,0,A1
2,1,2025-02-21,12,0,A1
3,1,2025-02-22,9,8,A1
4,1,2025-02-23,4,7,A1


In [48]:
# --- Load activity_level for sensor-complete students ---
activity_frames = []
for sid in range(1, 36):
    if sid in sensor_incomplete:
        continue
    path = f"{BASE}/{sid}/activity_level.csv"
    if os.path.exists(path):
        df = pd.read_csv(path)
        df['student_id'] = sid
        activity_frames.append(df)

activity_all = pd.concat(activity_frames, ignore_index=True)

# ISO 8601 timestamps with Z suffix → UTC date
activity_all['date'] = pd.to_datetime(activity_all['timestamp'], utc=True).dt.date

# Compute daily proportion of active minutes per student
active_levels = ['LIGHTLY_ACTIVE', 'FAIRLY_ACTIVE', 'VERY_ACTIVE']
activity_all['is_active'] = activity_all['level'].isin(active_levels)

daily_activity = (activity_all
    .groupby(['student_id', 'date'])
    .agg(total_minutes=('level', 'count'),
         active_minutes=('is_active', 'sum'))
    .reset_index())
daily_activity['proportion_active'] = daily_activity['active_minutes'] / daily_activity['total_minutes']

print(f"Activity: {len(daily_activity)} student-days from {daily_activity['student_id'].nunique()} students")
print("\nActivity level distribution (all records):")
print(activity_all['level'].value_counts())
daily_activity.head()

Activity: 2753 student-days from 32 students

Activity level distribution (all records):
level
SEDENTARY            1059736
LIGHTLY_ACTIVE        497944
MODERATELY_ACTIVE     123458
VERY_ACTIVE            85106
Name: count, dtype: int64


,student_id,date,total_minutes,active_minutes,proportion_active
0,1,2025-02-21,560,98,0.175000
1,1,2025-02-22,709,142,0.200282
2,1,2025-02-23,856,200,0.233645
3,1,2025-02-24,868,171,0.197005
4,1,2025-02-25,903,182,0.201550


In [49]:
# --- Load sleep for sensor-complete students ---
sleep_frames = []
for sid in range(1, 36):
    if sid in sensor_incomplete:
        continue
    path = f"{BASE}/{sid}/sleep.csv"
    if os.path.exists(path):
        df = pd.read_csv(path)
        df['student_id'] = sid
        sleep_frames.append(df)

sleep_all = pd.concat(sleep_frames, ignore_index=True)
sleep_all['date'] = pd.to_datetime(sleep_all['timestamp'], format='ISO8601', utc=True).dt.date
sleep_daily = sleep_all[['student_id', 'date', 'overall_score']].rename(columns={'overall_score': 'sleep_score'})
print(f"Sleep: {len(sleep_daily)} student-days from {sleep_daily['student_id'].nunique()} students")

# --- Clean oxygen: drop physiologically implausible SpO2 readings (<80%) ---
oxygen_frames = []
for sid in range(1, 36):
    if sid in sensor_incomplete:
        continue
    path = f"{BASE}/{sid}/oxygen.csv"
    if os.path.exists(path):
        df = pd.read_csv(path)
        df['student_id'] = sid
        oxygen_frames.append(df)

oxygen_all = pd.concat(oxygen_frames, ignore_index=True)
before = len(oxygen_all)
oxygen_all = oxygen_all[(oxygen_all['value'] >= 80) & (oxygen_all['value'] <= 100)]
after = len(oxygen_all)
print(f"\nOxygen outliers removed: {before - after:,} rows ({(before - after) / before:.1%} of total)")
print(f"Valid oxygen readings retained: {after:,}")

# --- Clean stress: drop rows where device calculation failed ---
stress_frames = []
for sid in range(1, 36):
    if sid in sensor_incomplete:
        continue
    path = f"{BASE}/{sid}/stress.csv"
    if os.path.exists(path):
        df = pd.read_csv(path)
        df['student_id'] = sid
        stress_frames.append(df)

stress_all = pd.concat(stress_frames, ignore_index=True)
before_s = len(stress_all)
stress_all = stress_all[stress_all['CALCULATION_FAILED'].astype(str).str.lower() != 'true']
after_s = len(stress_all)
print(f"\nStress CALCULATION_FAILED rows dropped: {before_s - after_s:,} ({(before_s - after_s) / before_s:.1%} of total)")

Sleep: 1829 student-days from 32 students

Oxygen outliers removed: 218,760 rows (22.4% of total)
Valid oxygen readings retained: 759,607

Stress CALCULATION_FAILED rows dropped: 341 (19.2% of total)


In [50]:
# --- Build unified per-day DataFrame ---
# Aggregate daily_questions to one row per student-day (some students have multiple surveys/day)
dq_daily = (dq_all
    .groupby(['student_id', 'date', 'course'])
    .agg(stress=('stress', 'mean'), anxiety=('anxiety', 'mean'))
    .reset_index())

# Merge activity and sleep onto the daily survey rows
daily = (dq_daily
    .merge(daily_activity[['student_id', 'date', 'proportion_active']], on=['student_id', 'date'], how='left')
    .merge(sleep_daily, on=['student_id', 'date'], how='left'))

print(f"Merged daily DataFrame: {daily.shape[0]} rows × {daily.shape[1]} columns")
print(f"Students: {daily['student_id'].nunique()} | Date range: {daily['date'].min()} to {daily['date'].max()}")
print(f"\nMissing values per column:")
print(daily.isnull().sum())
daily.head(10)

Merged daily DataFrame: 3139 rows × 7 columns
Students: 35 | Date range: 2025-02-14 to 2025-07-09

Missing values per column:
student_id              0
date                    0
course                  0
stress                  0
anxiety                 0
proportion_active     861
sleep_score          1610
dtype: int64


,student_id,date,course,stress,anxiety,proportion_active,sleep_score
0,1,2025-02-19,A1,15.0,0.0,NaN,NaN
1,1,2025-02-20,A1,33.0,0.0,NaN,NaN
2,1,2025-02-21,A1,12.0,0.0,0.175000,NaN
3,1,2025-02-22,A1,9.0,8.0,0.200282,NaN
4,1,2025-02-23,A1,4.0,7.0,0.233645,78.0
5,1,2025-02-24,A1,37.0,16.0,0.197005,85.0
6,1,2025-02-25,A1,34.0,13.0,0.201550,82.0
7,1,2025-02-26,A1,39.0,17.0,0.209302,84.0
8,1,2025-02-27,A1,22.0,32.0,0.248160,81.0
9,1,2025-02-28,A1,22.0,11.0,0.244613,84.0


**Data profile notes:**

- **Structure:** The dataset spans 35 students across 3 courses (A1, A2, B), each with up to 7 sensor and survey data streams stored in separate CSVs per student.
- **Incomplete participants:** Students 3, 12, and 14 have only `daily_questions.csv` and are excluded from any analyses requiring sensor data (activity, sleep, oxygen, stress).
- **Timestamp formats:** `daily_questions.csv` stores timestamps as Unix epoch seconds; all sensor files use ISO 8601 with a Z suffix. Both are standardized to UTC date for daily-level merging.
- **Oxygen outliers:** ~22% of raw SpO2 readings fell below 80%, which is physiologically implausible (likely sensor dropout). These were removed before any oxygen-based analysis.
- **Stress calculation failures:** A meaningful share of device-computed stress scores had `CALCULATION_FAILED = True` (e.g., 10/28 rows for Student 1, 30/42 for Student 31). These rows are dropped; only valid `STRESS_SCORE` values are used.
- **Merged DataFrame:** The final `daily` DataFrame has one row per student per day, with columns for course, self-reported stress, self-reported anxiety, proportion of active minutes, and sleep score. Missing values in `proportion_active` and `sleep_score` reflect days where no sensor data was recorded for that student.

---

## Section 3 — Analysis

Each question below includes the analysis code and results, a chart, and an interpretation of the findings.

### Question 1 — How does stress differ by course context?

How do average self-reported stress and anxiety scores compare across the three course groups (A1, A2, B)? This uses all 35 students since the question only requires `daily_questions.csv`.

In [51]:
# Average stress and anxiety per course group across all survey responses
q1_summary = (dq_all
    .groupby('course')[['stress', 'anxiety']]
    .agg(['mean', 'count'])
    .round(2))

q1_summary.columns = ['Avg Stress', 'Stress Responses', 'Avg Anxiety', 'Anxiety Responses']

print("Average self-reported stress and anxiety by course group (scale 0–100):")
q1_summary

Average self-reported stress and anxiety by course group (scale 0–100):


,Avg Stress,Stress Responses,Avg Anxiety,Anxiety Responses
course,,,,
A1,29.28,977,31.87,977
A2,27.49,767,30.16,767
B,39.18,1389,38.48,1389


**Interpretation — Q1:**

Course B stands out clearly: its average stress score (39.18) is roughly 10 points higher than both A1 (29.28) and A2 (27.49), and the same gap appears in anxiety (38.48 vs 31.87 and 30.16). A1 and A2 are closely matched to each other, which suggests the difference is driven by something specific to Course B — potentially heavier workload, different assessment structure, or a different student cohort — rather than a general upward drift across all courses.

This finding matters for interpreting Q2 and Q3: since Course B students report significantly higher stress, any behavioral patterns across the whole dataset may partly reflect the composition of course groups rather than pure within-student behavioral effects. Course B students experiencing higher stress are included in the same analysis as A1 and A2 students, which could obscure or distort the behavioral correlations tested next.

In [ ]:
import plotly.express as px

# Chart 1: Average stress and anxiety by course group (Q1)
q1_plot = dq_all.groupby('course')[['stress', 'anxiety']].mean().reset_index()
q1_plot = q1_plot.melt(id_vars='course', var_name='Measure', value_name='Average Score')
q1_plot['Measure'] = q1_plot['Measure'].str.capitalize()

fig1 = px.bar(
    q1_plot,
    x='course',
    y='Average Score',
    color='Measure',
    barmode='group',
    title='Course B Students Report ~10 Points Higher Stress and Anxiety Than A1 or A2',
    labels={
        'course': 'Course Group',
        'Average Score': 'Average Score (0–100)',
        'Measure': ''
    },
    color_discrete_sequence=['#E15759', '#4E79A7']
)
fig1.update_layout(yaxis_range=[0, 100], legend=dict(orientation='h', y=1.1))
fig1.show(renderer="png")
fig1.write_image("chart1_stress_by_course.png")

**Chart 1 — Grouped bar chart: Stress and anxiety by course group**

A grouped bar chart is the right choice because Q1 involves comparing two separate numeric measures (stress and anxiety) across three categorical groups (A1, A2, B). The side-by-side grouping makes both within-course and between-course comparisons readable at a glance. The y-axis is fixed to 0–100 to reflect the actual survey scale and prevent visually inflating the gap. The title states the finding directly rather than describing the data.

### Question 2 — Does being more active associate with lower stress?

On days when a student has a higher proportion of active minutes (LIGHTLY_ACTIVE, FAIRLY_ACTIVE, VERY_ACTIVE), do they also report lower stress scores? This uses sensor-complete students only (excluding 3, 12, 14) and only days where both activity and survey data are available.

In [52]:
# Filter to days where activity data exists
activity_days = daily.dropna(subset=['proportion_active']).copy()

# Bin proportion_active into 4 equal-sized groups (quartiles)
activity_days['activity_quartile'] = pd.qcut(
    activity_days['proportion_active'],
    q=4,
    labels=['Q1 (least active)', 'Q2', 'Q3', 'Q4 (most active)']
)

# Mean stress per activity quartile
q2_summary = (activity_days
    .groupby('activity_quartile', observed=True)['stress']
    .agg(avg_stress='mean', days='count')
    .round(2))

# Pearson correlation between proportion_active and stress
corr_activity = activity_days['proportion_active'].corr(activity_days['stress'])

print(f"Pearson correlation (proportion_active vs stress): r = {corr_activity:.3f}")
print(f"Days with activity + survey data: {len(activity_days)}\n")
q2_summary

Pearson correlation (proportion_active vs stress): r = 0.082
Days with activity + survey data: 2278



,avg_stress,days
activity_quartile,,
Q1 (least active),32.91,570
Q2,36.82,569
Q3,37.38,569
Q4 (most active),36.23,570


**Interpretation — Q2:**

The Pearson correlation is r = +0.082 — weakly positive and near zero, meaning more active days do not associate with lower stress. The quartile table confirms this: the least active quartile (Q1) actually has the lowest average stress (32.91), while the more active quartiles (Q2–Q4) have slightly higher stress around 36–37. This is the opposite of the expected direction.

A likely explanation is reverse causality: busier, higher-stress days may naturally involve more movement (walking between classes, errands, campus activity), while lower-stress days might involve more sedentary rest. It is also possible that Course B students — who report higher stress overall — happen to have different activity patterns than A1/A2 students, confounding the relationship. Either way, same-day physical activity does not appear to be a meaningful predictor of lower self-reported stress in this dataset.

### Question 3 — Does sleeping better associate with lower stress?

On days following a higher sleep overall score, do students report lower stress and anxiety? This uses the same sensor-complete students and compares the sleep–stress correlation against the activity–stress correlation from Q2 to identify the stronger behavioral predictor.

In [53]:
# Filter to days where sleep data exists
sleep_days = daily.dropna(subset=['sleep_score']).copy()

# Bin sleep_score into 4 equal-sized groups (quartiles)
sleep_days['sleep_quartile'] = pd.qcut(
    sleep_days['sleep_score'],
    q=4,
    labels=['Q1 (worst sleep)', 'Q2', 'Q3', 'Q4 (best sleep)']
)

# Mean stress per sleep quartile
q3_summary = (sleep_days
    .groupby('sleep_quartile', observed=True)['stress']
    .agg(avg_stress='mean', days='count')
    .round(2))

# Pearson correlation between sleep_score and stress
corr_sleep = sleep_days['sleep_score'].corr(sleep_days['stress'])

print(f"Pearson correlation (sleep_score vs stress):      r = {corr_sleep:.3f}")
print(f"Pearson correlation (proportion_active vs stress): r = {corr_activity:.3f}")
print(f"\nDays with sleep + survey data: {len(sleep_days)}\n")
q3_summary

Pearson correlation (sleep_score vs stress):      r = 0.018
Pearson correlation (proportion_active vs stress): r = 0.082

Days with sleep + survey data: 1529



,avg_stress,days
sleep_quartile,,
Q1 (worst sleep),36.56,415
Q2,36.13,420
Q3,38.36,314
Q4 (best sleep),35.37,380


**Interpretation — Q3:**

The Pearson correlation between sleep score and stress is r = +0.018 — essentially zero. The quartile table shows no meaningful trend: stress levels across the four sleep quartiles (36.56, 36.13, 38.36, 35.37) are nearly flat, with no consistent decrease as sleep quality improves.

This is a null result: sleep quality on a given night does not associate with same-day self-reported stress in this dataset. One reason may be that sleep effects are lagged — yesterday's sleep score may better predict today's stress than today's score does. Another is that the wearable sleep score (a composite device metric) may not capture the dimensions of sleep most relevant to subjective stress, such as dreams, perceived restfulness, or wake time. Comparing both correlations directly — r = 0.082 for activity and r = 0.018 for sleep — neither behavior is a meaningful same-day predictor, and the central question of which behavior "matters more" cannot be answered from this data without further investigation into lagged or within-person effects.

In [ ]:
# Chart 2: Stress by activity vs sleep quartile (Q2 + Q3)
act_plot = (activity_days
    .groupby('activity_quartile', observed=True)['stress']
    .mean().reset_index())
act_plot.columns = ['quartile', 'avg_stress']
act_plot['quartile_num'] = range(1, 5)
act_plot['predictor'] = 'Physical Activity'

slp_plot = (sleep_days
    .groupby('sleep_quartile', observed=True)['stress']
    .mean().reset_index())
slp_plot.columns = ['quartile', 'avg_stress']
slp_plot['quartile_num'] = range(1, 5)
slp_plot['predictor'] = 'Sleep Quality'

combined = pd.concat([act_plot, slp_plot], ignore_index=True)

fig2 = px.line(
    combined,
    x='quartile_num',
    y='avg_stress',
    color='predictor',
    markers=True,
    title='Neither Daily Activity nor Sleep Quality Shows a Clear Same-Day Stress Association',
    labels={
        'quartile_num': 'Quartile (1 = lowest, 4 = highest)',
        'avg_stress': 'Average Stress Score (0–100)',
        'predictor': 'Daily Behavior'
    },
    color_discrete_sequence=['#F28E2B', '#76B7B2']
)
fig2.update_xaxes(tickvals=[1, 2, 3, 4], ticktext=['Q1 (low)', 'Q2', 'Q3', 'Q4 (high)'])
fig2.show(renderer="png")
fig2.write_image("chart2_stress_by_behavior_quartile.png")

print("Charts saved: chart1_stress_by_course.png, chart2_stress_by_behavior_quartile.png")

**Chart 2 — Line chart: Average stress by behavioral quartile (Q2 + Q3)**

A line chart with markers is used because the x-axis represents ordered quartiles — there is a meaningful direction from low to high behavior — and the slope (or lack of one) is itself the finding. Plotting both behavioral predictors on the same axes makes the comparison direct without requiring separate figures. The flat, irregular trajectories visually confirm what the Pearson correlations showed numerically: r = +0.082 for activity and r = +0.018 for sleep. Reporting a null result clearly is an act of professional judgment — the finding is that same-day measurement is insufficient, and a lagged design would be needed to answer the original question.

---

## Section 4 — Conclusions

**Summary of findings:**

The clearest finding is from Q1: Course B students report substantially higher stress (39.18) and anxiety (38.48) than students in A1 (stress: 29.28) or A2 (stress: 27.49), a gap of roughly 10 points that holds for both measures. This suggests course context is a meaningful driver of student stress, independent of individual behavior. Q2 and Q3 produced null results: neither same-day physical activity (r = +0.082) nor same-day sleep quality (r = +0.018) showed a meaningful association with lower self-reported stress. Surprisingly, the least active days were associated with slightly lower stress — the opposite of the expected direction — which likely reflects reverse causality, where high-stress days involve more movement rather than more movement reducing stress. The central question of which behavior better predicts stress could not be answered from same-day correlations alone; a lagged analysis would be a necessary next step.

**What surprised me:** Both behavioral correlations were near zero and weakly positive rather than negative. I expected sleep quality in particular to show at least a modest negative association with stress, given the well-established link in prior research. The flat quartile tables and near-zero r values suggest same-day measurement may be the wrong time window — the effect of sleep on stress may show up the following day rather than the same day.

**What I would investigate next:** The most important next step is a lagged analysis: does sleep score on day N predict stress on day N+1? I would also run the behavioral analysis separately within each course group to see whether the Course B effect is masking real within-group patterns. Exploring HRV data as an objective physiological stress signal (rather than the self-reported survey) would also strengthen the analysis.

**Limitations:** Self-reported stress scores from brief surveys are subject to response bias and may not capture the full complexity of stress. The sample is 35 students from a single institution across 3 courses, limiting generalizability. Same-day correlation cannot establish causation, and the time-granularity of the merge (daily averages) may smooth over within-day variation that matters. Sensor data gaps — students 3, 12, 14 had no sensor files, and ~22% of oxygen readings were invalid — mean behavioral analyses are based on a subset of available student-days.

---

## Competency Claim

In a `week6.md` file in your GitHub repository, write a short competency claim (2–4 sentences) for each domain you feel this project demonstrates. Be specific — cite something you actually did in this notebook.

Domains covered by this project:
- **C1 — Vibecoding and rapid prototyping** (used AI to plan the data cleaning pipeline before implementation; validated each step against actual output before proceeding)
- **C3 — Data handling** (loaded 35 student folders across 7 file types, handled two timestamp formats, filtered oxygen outliers, dropped CALCULATION_FAILED rows, merged streams into a unified per-day DataFrame)
- **C5 — Visualization** (grouped bar chart showing Course B stress gap; line chart showing flat behavioral correlations with actual r values cited in title and rationale)
- **C7 — Critical evaluation and professional judgment** (recognized and reported null results honestly, explained reverse causality as an alternative interpretation, flagged limitations of same-day correlation design)

---

## Section 5 — Process Reflection

**Why this dataset and topic:**
I chose this dataset because stress in college is something I experienced personally and wanted to understand better. My initial questions were whether sleep quality is associated with stress, and whether stress and anxiety levels are related — both things I found myself wondering about during my own time as a student. Finding a dataset with both wearable sensor data and self-reported surveys from real students felt like a meaningful match for those questions.

**Finding and understanding the data:**
The SSAQS dataset was downloaded as a ZIP archive and extracted locally. Before any code was written, I had to figure out what was actually inside it — 35 student folders, each with up to 7 separate CSV files. I used Claude to help map out the cleaning plan: identifying that students 3, 12, and 14 had no sensor data, recognizing two different timestamp formats (Unix epoch seconds in `daily_questions.csv` vs ISO 8601 in all sensor files), and deciding which of the 7 file types were actually relevant to my questions. That planning conversation shaped the entire structure of Section 2.

**What the AI revealed that I didn't expect:**
The biggest surprise came after the analysis ran. I expected sleep quality and physical activity to show a negative correlation with stress — more sleep and more activity meaning less stress — which is what prior research generally suggests. Instead, both correlations were near zero and slightly positive (r = +0.082 for activity, r = +0.018 for sleep). The AI helped me recognize this as a likely reverse causality problem: high-stress days may naturally involve more movement (walking, errands, classes), and the same-day measurement window may simply be the wrong time frame for detecting a sleep effect. That pivot — from "which behavior predicts lower stress" to "neither behavior shows a same-day association, and here's why" — changed the whole framing of the conclusions and the chart titles.

**What I would have explored if the dataset allowed it:**
One thing I was curious about going in was whether stress and anxiety levels are related to academic performance. The dataset doesn't include grade or assignment data, so that question couldn't be answered here. It remains something I'd want to investigate with a richer dataset that links behavioral and survey signals to actual course outcomes.